## 커뮤니티 데이터


### 네이버 종목토론 크롤링


In [51]:
import json
import warnings

import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.filterwarnings(
    "ignore",
    category=MarkupResemblesLocatorWarning,
)

In [52]:
# 검색 조건 설정
stock_code = "005930"
page_size = 100
timeout = 10

# 네이버 모바일 종목 토론방 검색 주소
list_api = "https://m.stock.naver.com/front-api/discussion/list"

# 일반 웹 브러우저 요청처럼 보이도록 설정
header = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
    )
}

# 여러 요청에서 동일한 헤더를 사용
session = requests.Session()
session.headers.update(header)

In [53]:
# API 요청에 전달할 검색 조건
list_params = {
    "discussionType": "domesticStock",
    "itemCode": stock_code,
    "pageSize": page_size,
    "isHolderOnly": "false",
    "excludesItemNews": "false",
    "isItemNewsOnly": "false",
}

# 토론방 목록 API 요청
response = session.get(
    list_api,
    params=list_params,
    timeout=timeout,
)

# HTTP 요청 실패 시 예외 발생
response.raise_for_status()

# 응답 데이터를 JSON으로 변환
list_payload = response.json()
if not list_payload.get("isSuccess"):
    error_message = list_payload.get("message") or "알 수 없는 오류"

    raise RuntimeError(
        f"종목 토론방 목록 조회 실패: 종목코드={stock_code}, 오류={error_message}"
    )

# 응답 결과에서 게시글 목록 추출
list_result = list_payload.get("result") or {}
raw_posts = list_result.get("posts") or []

raw_posts[0]

{'id': '427078050',
 'orderNo': '-427078050',
 'discussionType': 'domesticStock',
 'itemCode': '005930',
 'itemName': '삼성전자',
 'postType': 'normal',
 'writer': {'profileId': '28660336065406871',
  'profileType': 'normal',
  'nickname': '주식재밌어',
  'imageUrl': 'https://ssl.pstatic.net/imgstock/fn/real/_front/image/profile/avatar-8.png',
  'isHolderVerified': False,
  'isNiConnected': False,
  'isRequester': False},
 'writtenAt': '2026-08-03T20:34:18',
 'title': '벼락거지 됐는데 ',
 'contentSwReplaced': '민주당 좋다고 또 찍어주지 ㅋ 호되게 당해봐야 알지',
 'contentSwReplacedButImg': '민주당 좋다고 또 찍어주지 ㅋ 호되게 당해봐야 알지',
 'contentJsonSwReplaced': '{"document": {"di": {"dif": false, "dio": [{"dia": {"p": 0, "t": 0, "sk": 20, "st": 29}, "dis": "N"}]}, "id": "01KZ3PDF82FSKVPBNAYF66HM6Q", "theme": "default", "version": "2.10.2", "language": "ko-KR", "components": [{"id": "SE-21c52931-04e5-435e-b25a-4bf5f7adce62", "value": [{"id": "SE-ae9bd03b-e828-444c-be72-64106ffe528e", "nodes": [{"id": "SE-89272301-0a38-430f-ba9e-98a11a3ed9

In [54]:
posts = []

for raw_post in raw_posts:
    # 게시글 기본 정보 추출
    post_id = raw_post.get("id") or ""

    item_code = raw_post.get("itemCode") or ""
    item_name = raw_post.get("itemName") or ""

    written_at = raw_post.get("writtenAt") or ""
    title = raw_post.get("title") or ""

    content_html = raw_post.get("contentSwReplaced") or ""
    content_soup = BeautifulSoup(content_html, "html.parser")

    # 스티커·이모티콘 이미지 제거
    for image in content_soup.select("img"):
        image.decompose()

    # <br>을 줄바꿈으로 변환하고 나머지 HTML 태그 제거
    content_text = content_soup.get_text("\n", strip=True)

    post_url = (
        f"https://m.stock.naver.com/domestic/stock/{item_code}/discussion/{post_id}"
    )

    # 본문 JSON에서 이미지 주소 추출
    images = []
    content_json_text = raw_post.get("contentJsonSwReplaced") or ""

    if content_json_text:
        try:
            content_json = json.loads(content_json_text)
            stack = [content_json]

            # 중첩된 JSON 구조를 순회
            while stack:
                value = stack.pop()

                if isinstance(value, dict):
                    if value.get("@ctype") == "image" and value.get("src"):
                        images.append(value["src"])

                    stack.extend(value.values())

                elif isinstance(value, list):
                    stack.extend(value)

        except json.JSONDecodeError as error:
            print(
                f"본문 JSON 해석 실패: 게시글 ID={raw_post.get('id') or ''}, 오류={error}"
            )

    # 중복된 이미지 주소 제거
    images = list(dict.fromkeys(images))

    # 필요한 정보만 하나의 딕셔너리로 저장
    posts.append(
        {
            "item_code": item_code,
            "item_name": item_name,
            "written_at": written_at,
            "title": title,
            "content_text": content_text,
            "content_images": images,
            "post_url": post_url,
        }
    )

posts[0]

{'item_code': '005930',
 'item_name': '삼성전자',
 'written_at': '2026-08-03T20:34:18',
 'title': '벼락거지 됐는데 ',
 'content_text': '민주당 좋다고 또 찍어주지 ㅋ 호되게 당해봐야 알지',
 'content_images': [],
 'post_url': 'https://m.stock.naver.com/domestic/stock/005930/discussion/427078050'}

In [55]:
# 수집한 게시글을 JSONL 파일로 저장
output_path = "community.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    file.writelines(json.dumps(post, ensure_ascii=False) + "\n" for post in posts)

print(f"저장 완료: {output_path}")
print(f"저장된 게시글 수: {len(posts)}개")

저장 완료: community.jsonl
저장된 게시글 수: 100개
